#  02) Find Gaia stars in all DDF

- creation date : 2026-04-08
- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS

## 1. Imports & configuration

In [ ]:
import requests
import pandas as pd
import numpy as np
import json
import os
import time
import matplotlib.pyplot as plt
import warnings
from astroquery.gaia import Gaia

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

```
| **Name of Field**    | **RA(Degrees)** | **Dec (Degress)**| **Type**          |
| -------------------- | --------------- | ---------------- | ----------------- |
| **Carina**           | 161.5           | -59.7            | Galaxie/Nébuleuse |
| **ELAIS-S1**         | 9.45            | -44.0            | DDF               |
| **COSMOS**           | 150.1           | +2.2             | DDF               |
| **Trifid-Lagoon**    | 270.5           | -23.0            | Nébuleuse         |
| **M49**              | 187.4           | +8.0             | Galaxie           |
| **Rubin_SV_280_-48** | 280.0           | -48.0            | Test SV           |
| **Rubin_SV_320_-15** | 320.0           | -15.0            | Test SV           |
| **Rubin_SV_216_-17** | 216.0           | -17.0            | Test SV           |
| **Rubin_SV_212_-7**  | 212.0           | -7.0             | Test SV           |
| **Rubin_SV_225_-40** | 225.0           | -40.0            | Test SV           |
```

### DDF definition

In [ ]:
CONE_RADIUS = 1800.0  # Cone search radius in arcsec (0.5 deg per DDF)

BANDS = list("ugrizy")

# LSST Deep Drilling Fields (RA/Dec J2000)
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "XMM-LSS": (35.7080, -4.750),
    "ECDFS": (53.1250, -27.8),
    "EDFS-a": (58.9, -49.315),
    "EDFS-b": (63.6, -47.6),
    "EDFS": (61.24, -48.423),
    "M49": (187.4, 8.0),
}

In [ ]:
def generate_gaia_query(ra, dec, radius=1.8, mag_limit=21, flux_over_error_limit=5, ruwe_limit=1.4):
    """
    Génère une requête ADQL pour interroger Gaia DR3 autour de (ra, dec).

    Paramètres :
    -----------
    ra : float
        Ascension droite du centre de la région (en degrés).
    dec : float
        Déclinaison du centre de la région (en degrés).
    radius : float, optionnel
        Rayon de la région de recherche (en degrés, par défaut 1.5).
    mag_limit : float, optionnel
        Limite supérieure de la magnitude (par défaut 21).
    flux_over_error_limit : float, optionnel
        Limite inférieure pour phot_g_mean_flux_over_error (par défaut 5).
    ruwe_limit : float, optionnel
        Limite supérieure pour RUWE (par défaut 1.4).

    Retourne :
    --------
    str
        Chaîne de caractères contenant la requête ADQL.
    """
    query = f"""
SELECT
    g.source_id,
    g.ra, g.dec,
    g.phot_g_mean_mag,
    g.astrometric_params_solved,
    g.astrometric_gof_al,
    g.phot_bp_rp_excess_factor,
    v.mean_mag_g_fov,
    v.median_mag_g_fov,
    v.std_dev_mag_g_fov,
    v.in_vari_long_period_variable,
    v.in_vari_eclipsing_binary,
    v.in_vari_classification_result,
    v.in_vari_rrlyrae,
    v.in_vari_cepheid,
    v.in_vari_planetary_transit,
    v.in_vari_short_timescale,
    v.in_vari_long_period_variable,
    v.in_vari_eclipsing_binary,
    v.in_vari_rotation_modulation,
    v.in_vari_ms_oscillator,
    v.in_vari_agn,
    v.in_vari_microlensing,
    v.in_vari_compact_companion
FROM gaiadr3.gaia_source AS g
LEFT JOIN gaiadr3.vari_summary AS v ON g.source_id = v.source_id
WHERE
    1 = CONTAINS(
        POINT('ICRS', g.ra, g.dec),
        CIRCLE('ICRS', {ra}, {dec}, {radius})
    )
    AND g.phot_g_mean_mag < {mag_limit}
    AND g.phot_g_mean_flux_over_error > {flux_over_error_limit}
    AND g.ruwe < {ruwe_limit}
"""
    return query

In [ ]:
def filter_stable_point_sources(
    df, std_dev_threshold=0.05, bp_rp_excess_min=1, bp_rp_excess_max=2, gof_al_threshold=3
):
    """
    Filtre un DataFrame Gaia pour ne garder que les sources ponctuelles et stables.

    Paramètres :
    -----------
    df : pandas.DataFrame
        DataFrame contenant les données Gaia (doit inclure les colonnes in_vari_*, std_dev_mag_g_fov,
        astrometric_params_solved, astrometric_gof_al, phot_bp_rp_excess_factor).
    std_dev_threshold : float, optionnel
        Seuil pour l'écart-type de la magnitude (par défaut 0.05).
    bp_rp_excess_min : float, optionnel
        Limite inférieure pour phot_bp_rp_excess_factor (par défaut 1).
    bp_rp_excess_max : float, optionnel
        Limite supérieure pour phot_bp_rp_excess_factor (par défaut 2).
    gof_al_threshold : float, optionnel
        Seuil pour astrometric_gof_al (par défaut 3).

    Retourne :
    --------
    pandas.DataFrame
        DataFrame filtré contenant uniquement les sources ponctuelles stables.
    """

    # 1. Masque pour les sources sans variabilité détectée
    stable_mask = (
        (df["in_vari_classification_result"].fillna(0) == 0)
        & (df["in_vari_rrlyrae"].fillna(0) == 0)
        & (df["in_vari_cepheid"].fillna(0) == 0)
        & (df["in_vari_planetary_transit"].fillna(0) == 0)
        & (df["in_vari_short_timescale"].fillna(0) == 0)
        & (df["in_vari_long_period_variable"].fillna(0) == 0)
        & (df["in_vari_eclipsing_binary"].fillna(0) == 0)
        & (df["in_vari_rotation_modulation"].fillna(0) == 0)
        & (df["in_vari_ms_oscillator"].fillna(0) == 0)
        & (df["in_vari_agn"].fillna(0) == 0)
        & (df["in_vari_microlensing"].fillna(0) == 0)
        & (df["in_vari_compact_companion"].fillna(0) == 0)
    )

    # 2. Masque pour la stabilité photométrique
    photometric_stability_mask = df["std_dev_mag_g_fov"].fillna(0) < std_dev_threshold

    # 3. Masques pour les sources ponctuelles
    point_source_mask_params = df["astrometric_params_solved"] == 31
    point_source_mask_gof = df["astrometric_gof_al"] < gof_al_threshold
    point_source_mask_excess = (df["phot_bp_rp_excess_factor"] > bp_rp_excess_min) & (
        df["phot_bp_rp_excess_factor"] < bp_rp_excess_max
    )
    point_source_mask = point_source_mask_params & point_source_mask_gof & point_source_mask_excess

    # 4. Combiner tous les masques
    stable_point_sources_df = df[stable_mask & photometric_stability_mask & point_source_mask]

    print(f"Nombre de sources ponctuelles stables : {len(stable_point_sources_df)}")

    return stable_point_sources_df

In [ ]:
def plot_sources_ra_dec(
    df,
    magnitude_col="phot_g_mean_mag",
    size_scale=100,
    color_stable="blue",
    color_variable="red",
    title="Carte des sources (RA, Dec)",
    figsize=(12, 8),
    save_path=None,
):
    """
    Trace les sources en coordonnées (RA, Dec) avec une taille liée à la magnitude.

    Paramètres :
    -----------
    df : pandas.DataFrame
        DataFrame contenant les données des sources (doit inclure ra, dec, et la colonne de magnitude).
    magnitude_col : str, optionnel
        Nom de la colonne contenant la magnitude (par défaut 'phot_g_mean_mag').
    size_scale : float, optionnel
        Facteur d'échelle pour la taille des points (par défaut 100).
    color_stable : str, optionnel
        Couleur des sources stables (par défaut 'blue').
    color_variable : str, optionnel
        Couleur des sources variables (par défaut 'red').
    title : str, optionnel
        Titre du graphique (par défaut "Carte des sources (RA, Dec)").
    figsize : tuple, optionnel
        Taille de la figure (par défaut (12, 8)).
    save_path : str, optionnel
        Chemin pour sauvegarder le graphique (par défaut None, ne sauvegarde pas).
    """

    # Vérifier que les colonnes nécessaires sont présentes
    required_cols = ["ra", "dec", magnitude_col]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"La colonne '{col}' est manquante dans le DataFrame.")

    # Créer la figure
    plt.figure(figsize=figsize)

    # Inverser la magnitude pour que les étoiles brillantes soient plus grosses
    # (plus la magnitude est faible, plus l'étoile est brillante)
    sizes = size_scale * 10 ** (-0.4 * df[magnitude_col]) * 1e5

    # Tracer les sources stables (exemple : celles sans variabilité)
    # (Adapte cette condition selon tes critères de stabilité)
    stable_mask = (df["in_vari_classification_result"].fillna(0) == 0) & (
        df["in_vari_eclipsing_binary"].fillna(0) == 0
    )
    plt.scatter(
        df[~stable_mask]["ra"],
        df[~stable_mask]["dec"],
        s=sizes[~stable_mask],
        c=color_variable,
        alpha=0.6,
        label="Variables",
    )
    plt.scatter(
        df[stable_mask]["ra"],
        df[stable_mask]["dec"],
        s=sizes[stable_mask],
        c=color_stable,
        alpha=0.6,
        label="Stables",
    )

    # Ajouter des labels et un titre
    plt.xlabel("RA (degrees)")
    plt.ylabel("Dec (degrees)")
    plt.title(title)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()

    # Inverser l'axe des x pour que RA augmente de droite à gauche (convention astronomique)
    plt.gca().invert_xaxis()

    # Sauvegarder le graphique si un chemin est fourni
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    # Afficher le graphique
    plt.show()

## Loop on Deep  field

In [ ]:
# Boucle sur les champs profonds
for field_name, (ra, dec) in DEEP_FIELDS.items():
    print(f"\nTraitement du champ {field_name} (RA={ra}, Dec={dec})...")

    # Générer la requête pour ce champ
    query = generate_gaia_query(ra, dec, radius=1.5, mag_limit=21)

    try:
        # Exécuter la requête
        job = Gaia.launch_job(query)
        df = job.get_results().to_pandas()

        # Filtrer les sources ponctuelles stables
        stable_sources_df = filter_stable_point_sources(df)

        # Sauvegarder les résultats dans un fichier CSV
        output_filename = f"stable_sources_{field_name}.csv"
        stable_sources_df.to_csv(output_filename, index=False)
        print(f"Résultats sauvegardés dans {output_filename}")

        plot_sources_ra_dec(
            df,
            # stable_sources_df,
            magnitude_col="phot_g_mean_mag",
            size_scale=20,  # Ajuste ce paramètre pour changer la taille des points
            color_stable="blue",
            color_variable="red",
            title=f"Gaia map sources in {field_name} region ({ra:.2f}, {dec:.2f})",
            save_path=f"sources_{field_name}_ra_dec.png",  # Optionnel : sauvegarde le graphique
        )

    except Exception as e:
        print(f"Erreur lors du traitement du champ {field_name}: {e}")